In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt

# === Hiperparâmetros ===
STATE_SIZE = 1
ACTION_SIZE = 1
GAMMA = 0.99
ACTOR_LR = 1e-4
CRITIC_LR = 1e-3
BATCH_SIZE = 64
MEMORY_SIZE = 100000
TAU = 0.005
MAX_STEPS = 200
MAX_EPISODES = 1000
ACTION_BOUND = 2.0
ACTION_EFFECT = 80
NOISE_STD = 0.05
rewards_history = []
erro_final_por_ep = []

# === Parâmetros MBRL (mantidos simples e derivados dos existentes) ===
ROLLOUT_HORIZON = 5          # comprimento dos rollouts do modelo
MODEL_TRAIN_STEPS = 200      # passos de treinamento do modelo por episódio
MODEL_BATCH_SIZE = 128       # batch para treinar o modelo
MODEL_LR = CRITIC_LR         # taxa de aprendizado do modelo

# === Replay Buffer ===
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))
    def __len__(self):
        return len(self.buffer)

# === Redes do DDPG (mantidas) ===
class Actor(nn.Module):
    def __init__(self, state_size, action_size):
        super(Actor, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_size),
            nn.Tanh()
        )
    def forward(self, x):
        return self.fc(x) * ACTION_BOUND

class Critic(nn.Module):
    def __init__(self, state_size, action_size):
        super(Critic, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size + action_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        return self.fc(x)

# === Modelo Dinâmico (aprende s' dado s, a) ===
class DynamicsModel(nn.Module):
    def __init__(self, state_size, action_size):
        super(DynamicsModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size + action_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, state_size)
        )
    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        delta = self.net(x)  # aprende o delta do estado
        next_state_pred = state + delta
        return next_state_pred

# === Inicialização ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

actor = Actor(STATE_SIZE, ACTION_SIZE).to(device)
critic = Critic(STATE_SIZE, ACTION_SIZE).to(device)
target_actor = Actor(STATE_SIZE, ACTION_SIZE).to(device)
target_critic = Critic(STATE_SIZE, ACTION_SIZE).to(device)
dynamics = DynamicsModel(STATE_SIZE, ACTION_SIZE).to(device)

target_actor.load_state_dict(actor.state_dict())
target_critic.load_state_dict(critic.state_dict())

actor_optimizer = optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic_optimizer = optim.Adam(critic.parameters(), lr=CRITIC_LR)
model_optimizer = optim.Adam(dynamics.parameters(), lr=MODEL_LR)

memory = ReplayBuffer(MEMORY_SIZE)
losses = []

# Buffer auxiliar para treinar o modelo (mesma estrutura)
model_memory = ReplayBuffer(MEMORY_SIZE)

# === Funções auxiliares ===
def select_action(state, noise=NOISE_STD):
    state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
    action = actor(state_t).cpu().data.numpy().flatten()
    action += noise * np.random.randn(ACTION_SIZE)
    return np.clip(action, -ACTION_BOUND, ACTION_BOUND)

def soft_update(target, source, tau):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

def calculate_reward(erro_lateral):
    normalized_error = abs(erro_lateral) / 100
    reward = 1.0 - normalized_error**2
    return reward

# === Treino do modelo dinâmico ===
def train_dynamics():
    if len(model_memory) < MODEL_BATCH_SIZE:
        return None
    states, actions, _, next_states, _ = model_memory.sample(MODEL_BATCH_SIZE)
    states = torch.FloatTensor(states).to(device)
    actions = torch.FloatTensor(actions).to(device)
    next_states = torch.FloatTensor(next_states).to(device)

    next_pred = dynamics(states, actions)
    loss_model = nn.MSELoss()(next_pred, next_states)

    model_optimizer.zero_grad()
    loss_model.backward()
    model_optimizer.step()
    return loss_model.item()

# === Geração de rollouts sintéticos com o modelo ===
def generate_model_rollouts(n_rollouts=MODEL_BATCH_SIZE, horizon=ROLLOUT_HORIZON):
    if len(memory) == 0:
        return
    # Amostra estados reais como seeds
    real_states, _, _, _, _ = memory.sample(min(n_rollouts, len(memory)))
    for s in real_states:
        state = np.array(s)  # shape (1,)
        for t in range(horizon):
            # Seleciona ação pela política atual (sem ruído adicional)
            action = select_action(state, noise=0.0)
            # Prediz próximo estado com o modelo
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            action_t = torch.FloatTensor(action).unsqueeze(0).to(device)
            with torch.no_grad():
                next_state_t = dynamics(state_t, action_t)
            next_state = next_state_t.cpu().numpy().flatten()
            # Clipes como no ambiente
            next_state[0] = np.clip(next_state[0], -100, 100)

            # Calcula recompensa e done com mesma regra
            erro_lateral = next_state[0]
            reward = calculate_reward(erro_lateral)
            done = abs(erro_lateral) > 100

            # Empurra transição sintética no replay buffer de treino
            memory.push(state, action, reward, next_state, done)

            state = next_state
            if done:
                break

# === Função de treinamento (DDPG com dados reais + sintéticos) ===
def train_step():
    if len(memory) < BATCH_SIZE:
        return None
    states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

    states = torch.FloatTensor(states).to(device)
    actions = torch.FloatTensor(actions).to(device)
    rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
    next_states = torch.FloatTensor(next_states).to(device)
    dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

    next_actions = target_actor(next_states)
    target_q = target_critic(next_states, next_actions)
    y = rewards + GAMMA * target_q * (1 - dones)
    critic_loss = nn.MSELoss()(critic(states, actions), y.detach())

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    actor_loss = -critic(states, actor(states)).mean()
    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    soft_update(target_actor, actor, TAU)
    soft_update(target_critic, critic, TAU)

    return critic_loss.item(), actor_loss.item()

# === Ambiente de simulação (real) ===
def simulate_step(state, action):
    erro_lateral = state[0]
    erro_lateral += float(action) * ACTION_EFFECT
    erro_lateral += random.uniform(-5, 5)
    erro_lateral = np.clip(erro_lateral, -100, 100)
    return np.array([erro_lateral])

# === Loop de interação ===
for episode in tqdm(range(MAX_EPISODES), desc="Treinando episódios"):
    state = np.array([random.uniform(-100, 100)])
    total_reward = 0

    # Coleta experiência real
    for step in range(MAX_STEPS):
        action = select_action(state)
        next_state = simulate_step(state, action)
        erro_lateral = next_state[0]

        reward = calculate_reward(erro_lateral)
        done = abs(erro_lateral) > 100

        # Guarda no buffer real e também para treino do modelo
        memory.push(state, action, reward, next_state, done)
        model_memory.push(state, action, reward, next_state, done)

        # Treina o modelo dinâmico algumas iterações por episódio
        for _ in range(MODEL_TRAIN_STEPS // MAX_STEPS):
            train_dynamics()

        # Gera rollouts sintéticos curtos para enriquecer o buffer
        generate_model_rollouts(n_rollouts=MODEL_BATCH_SIZE // 4, horizon=ROLLOUT_HORIZON)

        # Atualiza ator e crítico com batch misto (real + sintético)
        losses_val = train_step()
        if losses_val is not None:
            losses.append(losses_val)

        state = next_state
        total_reward += reward

        if done:
            break

    rewards_history.append(total_reward)
    erro_final_por_ep.append(state[0])
    print(f"[Ep {episode}] Reward: {total_reward:.2f} | Erro final: {state[0]:.2f}")

# === Salvar os modelos treinados ===
torch.save(actor.state_dict(), "mbrl_actor.pth")
torch.save(critic.state_dict(), "mbrl_critic.pth")
torch.save(dynamics.state_dict(), "mbrl_dynamics.pth")
print("✅ Modelos salvos (mbrl_actor.pth, mbrl_critic.pth, mbrl_dynamics.pth)")

# === Gráfico de recompensas ===
plt.figure(figsize=(10, 5))
plt.plot(rewards_history, label="Recompensa por episódio")
plt.xlabel("Episódio")
plt.ylabel("Recompensa total")
plt.title("Evolução da recompensa (MBRL + DDPG)")
plt.grid()
plt.legend()
plt.show()

# === Gráfico de erro final ===
plt.figure(figsize=(10, 5))
plt.plot(erro_final_por_ep, label="Erro lateral final")
plt.xlabel("Episódio")
plt.ylabel("Erro final")
plt.title("Convergência do erro lateral ao longo dos episódios (MBRL)")
plt.grid()
plt.legend()
plt.show()

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd

# === Hiperparâmetros ===
STATE_SIZE = 1
ACTION_SIZE = 1
ACTION_LIMIT = 0.9
ERRO_MAX = 300.0  # usado para normalização e visualização

# === Rede Actor (MBRL usa mesmo formato do DDPG/SAC) ===
class Actor(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super(Actor, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size),
            nn.Tanh()  # saída em [-1, 1]
        )
    def forward(self, x):
        return self.fc(x) * ACTION_LIMIT  # escala para [-0.9, 0.9]

# === Inicialização ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
actor = Actor(STATE_SIZE, ACTION_SIZE).to(device)

# Carregar modelo salvo (treinado com MBRL)
actor.load_state_dict(torch.load("mbrl_actor.pth", map_location=device))
actor.eval()
print("✅ Modelo MBRL carregado!")

# === Função para escolher ação ===
def select_action(state):
    state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
    with torch.no_grad():
        action = actor(state_t).cpu().numpy().flatten()
    return action[0]

# === Geração de dados ===
test_states = np.linspace(-ERRO_MAX, ERRO_MAX, 200).reshape(-1, 1)
action_list = [select_action(s) for s in test_states]

# === Exportar CSV ===
pd.DataFrame({
    "Erro_Lateral": test_states.flatten(),
    "Velocidade_Angular": action_list
}).to_csv("velocidade_predita_angular_mbrl.csv", index=False)
print("📁 CSV salvo como 'velocidade_predita_angular_mbrl.csv'")

# === Plot ===
plt.figure(figsize=(10, 5))
plt.plot(test_states, action_list, color='red', label="Velocidade Angular (MBRL actor)")
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel("Erro Lateral")
plt.ylabel(f"Velocidade Angular (±{ACTION_LIMIT})")
plt.title("Velocidade Angular vs Erro Lateral (MBRL actor)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("velocidade_vs_erro_angular_mbrl.png")
plt.show()
print("🖼️ Gráfico salvo como 'velocidade_vs_erro_angular_mbrl.png'")